In [2]:
import numpy as np

np.random.seed(42)

# 1. XOR 데이터 준비
# X Shape: (4, 2) -> 샘플 수 N=4, 입력 특성 수 D_in=2
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)

# y Shape: (4, 1) -> 샘플 수 N=4, 출력 정답 수 D_out=1
y = np.array([[0], [1], [1], [0]], dtype=float)

N = len(X)  # 미니배치 샘플 수 N = 4


# 2. Sigmoid 함수 정의
# 수식: sigma(x) = 1 / (1 + e^(-x))
# 도함수(미분): d_sigma(x) / dx = sigma(x) * (1 - sigma(x))
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# 3. 가중치 및 편향 초기화 (입력층 2 -> 은닉층 2 -> 출력층 1)
# 역전파의 기본 원칙: 어떤 행렬 M에 대한 기울기 dL/dM의 차원(Shape)은 반드시 원래 행렬 M과 같아야 함
W1 = np.array([ [ 4.0, 4.0], [4.0, 4.0] ]) # Shape: (2, 2) -> (입력 차원 2, 은닉 차원 2)
# W1 = np.random.randn(2, 2)
b1 = np.array([ [-2.0, -6.0] ]) # Shape: (1, 2) -> (1, 은닉 차원 2)
# b1 = np.zeros((1, 2))

W2 = np.array([ [ 4.0], [-8.0] ]) # Shape: (2, 1) -> (은닉 차원 2, 출력 차원 1)
# W2 = np.random.randn(2, 1)
b2 = np.array([ [-2.0] ]) # Shape: (1, 1) -> (1, 출력 차원 1)
# b2 = np.zeros((1, 1))

lr = 1.0  # 학습률 (Learning Rate)

# 4. 학습 루프
for epoch in range(10000):

    # =========================================================================
    # Forward Propagation (순전파)
    # =========================================================================

    # [은닉층 선형 결합] z1 = X @ W1 + b1
    # Shape: (4, 2) @ (2, 2) + (1, 2)[브로드캐스팅] = (4, 2)
    z1 = X @ W1 + b1

    # [은닉층 활성화] a1 = sigmoid(z1) (원소별 연산)
    # Shape: (4, 2)
    a1 = sigmoid(z1)

    # [출력층 선형 결합] z2 = a1 @ W2 + b2
    # Shape: (4, 1) = (4, 2) @ (2, 1) + (1, 1)[브로드캐스팅]
    z2 = a1 @ W2 + b2

    # [출력층 활성화 (예측값)] y_hat = sigmoid(z2) (원소별 연산)
    # Shape: (4, 1)
    y_hat = sigmoid(z2)

    # [손실 함수 (Binary Cross Entropy)]
    # 수식: L = - (1/N) * sum( y * ln(y_hat) + (1 - y) * ln(1 - y_hat) )
    # Shape: 스칼라 (단일 값)
    loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))

    # =========================================================================
    # Backpropagation (역전파)
    # =========================================================================

    # -------------------------------------------------------------------------
    # [Step 1] 출력층 선형 출력 z2에 대한 손실 미분 (dL_dz2)
    # -------------------------------------------------------------------------
    # * Chain Rule: dL/dz2 = (dL/dy_hat) * (dy_hat/dz2)
    #   1) dL/dy_hat = (1/N) * (y_hat - y) / (y_hat * (1 - y_hat))
    #   2) dy_hat/dz2 = y_hat * (1 - y_hat)  (Sigmoid 미분)
    #   3) 두 식을 곱하면 분모와 분자의 y_hat * (1 - y_hat) 항이 약분되어 소거됨
    # * 최종 수식: dL/dz2 = (y_hat - y) / N
    # * Shape: (4, 1) -> z2의 차원 (4, 1)과 동일
    dL_dz2 = (y_hat - y) / N

    # -------------------------------------------------------------------------
    # [Step 2] 출력층 가중치 W2와 편향 b2 미분 (dL_dW2, dL_db2)
    # -------------------------------------------------------------------------
    # [2-1. 가중치 W2 미분: dL_dW2]
    # * 순전파 수식: z2 = a1 @ W2 + b2  (원소: z2_{i,1} = sum_j(a1_{i,j} * W2_{j,1}) + b2_1)
    # * Chain Rule: dL/dW2_{j,1} = sum_{i=1}^N (dL/dz2_{i,1} * dz2_{i,1}/dW2_{j,1})
    #                            = sum_{i=1}^N (dL/dz2_{i,1} * a1_{i,j})
    # * 왜 Transpose(a1.T)를 사용하는가?
    #   1) 수식(합산) 관점: W2는 모든 샘플(i=1..4)의 계산에 공유되었으므로 샘플 인덱스 i를 기준으로
    #      내적(합산)해야 함. a1_{i,j}의 행과 열을 뒤집어 (a1.T)_{j,i}로 만들어야 행렬 곱으로 합산됨.
    #   2) 차원(Shape) 관점: 목표인 W2의 차원은 (2, 1)이고, 재료는 a1(4, 2)과 dL_dz2(4, 1)임.
    #      미니배치 차원(4)을 소거하려면 a1을 전치시켜 (2, 4) @ (4, 1) = (2, 1)로 맞춰야 함.
    dL_dW2 = a1.T @ dL_dz2

    # [2-2. 편향 b2 미분: dL_db2]
    # * Chain Rule: dL/db2 = sum_{i=1}^N (dL/dz2^{(i)} * dz2^{(i)}/db2) = sum_{i=1}^N (dL/dz2^{(i)} * 1)
    # * 순전파 때 b2(1, 1)가 4개 샘플에 브로드캐스팅되어 더해졌으므로, 역전파 시에는 샘플 차원(axis=0)으로
    #   오차를 모두 합산(np.sum)하며, keepdims=True로 원래 b2의 차원 (1, 1)을 유지함.
    dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

    # -------------------------------------------------------------------------
    # [Step 3] 은닉층 활성화 출력 a1 및 선형 출력 z1 미분 (dL_da1, dL_dz1)
    # -------------------------------------------------------------------------
    # [3-1. 은닉층 활성화 값 a1 미분: dL_da1]
    # * 순전파 수식: z2 = a1 @ W2 + b2
    # * Chain Rule: dL/da1_{i,j} = sum_k (dL/dz2_{i,k} * dz2_{i,k}/da1_{i,j})
    #                            = dL/dz2_{i,1} * W2_{j,1}
    # * 왜 Transpose(W2.T)를 사용하는가?
    #   1) 방향 관점: 순전파에서 은닉->출력(j->k) 방향으로 W2를 곱했으므로, 역전파에서 출력->은닉(k->j)
    #      방향으로 오차를 되돌려 분배하려면 가중치 행렬을 뒤집은 W2.T를 곱해야 함.
    #   2) 차원(Shape) 관점: 목표인 a1의 차원은 (4, 2)이고, 재료는 dL_dz2(4, 1)와 W2(2, 1)임.
    #      출력 차원(1)을 소거하고 은닉 차원(2)으로 확장하려면 W2를 전치시켜 (4, 1) @ (1, 2) = (4, 2)로 곱해야 함.
    dL_da1 = dL_dz2 @ W2.T

    # [3-2. 은닉층 선형 출력 z1 미분: dL_dz1]
    # * 순전파 수식: a1 = sigmoid(z1) (각 원소별로 독립 적용된 함수)
    # * Chain Rule: dL/dz1 = (dL/da1) * (da1/dz1)
    #   여기서 Sigmoid 미분에 의해 da1/dz1 = a1 * (1 - a1)
    # * 원소별 연산의 역전파이므로 행렬 곱(@)이 아닌 원소별 곱셈(*, Hadamard product)을 수행함.
    # * Shape: (4, 2) * (4, 2) * (4, 2) = (4, 2) -> z1의 차원 (4, 2)와 동일
    dL_dz1 = dL_da1 * a1 * (1 - a1)

    # -------------------------------------------------------------------------
    # [Step 4] 은닉층 가중치 W1과 편향 b1 미분 (dL_dW1, dL_db1)
    # -------------------------------------------------------------------------
    # [4-1. 가중치 W1 미분: dL_dW1]
    # * 순전파 수식: z1 = X @ W1 + b1  (원소: z1_{i,j} = sum_m(X_{i,m} * W1_{m,j}) + b1_j)
    # * 전체 Chain Rule 경로: dL/dW1 = (dL/dy_hat) * (dy_hat/dz2) * (dz2/da1) * (da1/dz1) * (dz1/dW1)
    #                               = dL_dz1과 dz1/dW1(= X)의 결합
    # * 원소별 수식: dL/dW1_{m,j} = sum_{i=1}^N (dL/dz1_{i,j} * dz1_{i,j}/dW1_{m,j})
    #                            = sum_{i=1}^N (X_{i,m} * dL/dz1_{i,j})
    # * 왜 Transpose(X.T)를 사용하는가?
    #   Step 2의 dL_dW2와 동일하게 4개 샘플(i=1..4)에 대한 기울기를 합산하고, W1의 차원 (2, 2)를 맞추기 위해
    #   X(4, 2)를 전치시켜 X.T(2, 4)로 만든 뒤 dL_dz1(4, 2)과 행렬 곱을 수행함: (2, 4) @ (4, 2) = (2, 2)
    dL_dW1 = X.T @ dL_dz1

    # [4-2. 편향 b1 미분: dL_db1]
    # * Chain Rule: dL/db1 = sum_{i=1}^N (dL/dz1^{(i)} * dz1^{(i)}/db1) = sum_{i=1}^N (dL/dz1^{(i)} * 1)
    # * 브로드캐스팅된 편향의 역전파이므로 샘플 차원(axis=0)으로 dL_dz1(4, 2)을 합산하여 (1, 2) 차원으로 만듦.
    dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

    # =========================================================================
    # Gradient Descent (경사 하강법을 통한 파라미터 업데이트)
    # =========================================================================
    # 각 파라미터와 해당 기울기(dL_dW, dL_db)의 Shape이 정확히 일치하므로 뺄셈 연산이 성립함
    W2 -= lr * dL_dW2  # (2, 1) -= 스칼라 * (2, 1)
    b2 -= lr * dL_db2  # (1, 1) -= 스칼라 * (1, 1)

    W1 -= lr * dL_dW1  # (2, 2) -= 스칼라 * (2, 2)
    b1 -= lr * dL_db1  # (1, 2) -= 스칼라 * (1, 2)

    if epoch % 2000 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

# 5. 최종 학습 결과 확인
print("\nX:")
print(X)
print("\n예측 결과:")
print(np.round(y_hat, 3))
print("\n분류 결과:")
print((y_hat >= 0.5).astype(int))
print("\nY:")
print(y)

Epoch 0, Loss: 0.2742
Epoch 2000, Loss: 0.0048
Epoch 4000, Loss: 0.0024
Epoch 6000, Loss: 0.0016
Epoch 8000, Loss: 0.0012

X:
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [1. 1.]]

예측 결과:
[[0.001]
 [0.999]
 [0.999]
 [0.001]]

분류 결과:
[[0]
 [1]
 [1]
 [0]]

Y:
[[0.]
 [1.]
 [1.]
 [0.]]
